# 01 · First model: predicting Premier League scores with a Poisson model

**Goal of this notebook:** download real match data, check that goals really behave like a Poisson distribution, fit the model, predict Big Six matches, and test honestly against the bookmakers.

Work through it top to bottom. Cells marked **🧠 Your turn** are for you to try before we move on.

## 0 · Setup (run every time you open the notebook)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
GITHUB_USER = "Sethchiji"          # ✏️ your GitHub username
REPO_NAME = "epl-score-predictor"

if IN_COLAB:
    repo_dir = f"/content/{REPO_NAME}"
    if not os.path.exists(repo_dir):
        !git clone -q https://github.com/{GITHUB_USER}/{REPO_NAME}.git {repo_dir}
    else:
        !git -C {repo_dir} pull -q
    %cd {repo_dir}
    !pip install -q -r requirements.txt
else:
    repo_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(repo_dir)

sys.path.insert(0, repo_dir)
print("Working in:", repo_dir)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import poisson

from src.data import load_recent, BIG_SIX, season_code, current_season_start
from src.model import fit_poisson
from src.evaluate import backtest, compare

pd.set_option("display.precision", 3)

## 1 · Load the data
We pull the current season plus the previous three from football-data.co.uk. Each row is one match: date, teams, full-time goals (`FTHG`/`FTAG`), result (`FTR` = H/D/A), and bookmaker odds.

In [ ]:
matches = load_recent(n_seasons=4)
print(matches.shape)
print(matches["season"].value_counts().sort_index())
matches.tail()

## 2 · Explore: do goals look Poisson?
The whole model rests on this assumption, so check it before trusting it. If goals are Poisson, the observed share of matches with 0, 1, 2… goals should match the Poisson curve with the same average.

In [ ]:
print(f"Average home goals: {matches.FTHG.mean():.2f}")
print(f"Average away goals: {matches.FTAG.mean():.2f}")
print("Result share:", matches.FTR.value_counts(normalize=True).round(3).to_dict())

goals = np.arange(0, 8)
check = pd.DataFrame({
    "goals": np.tile(goals, 2),
    "side": ["Home"] * len(goals) + ["Away"] * len(goals),
    "observed": np.concatenate([
        matches.FTHG.value_counts(normalize=True).reindex(goals, fill_value=0).values,
        matches.FTAG.value_counts(normalize=True).reindex(goals, fill_value=0).values]),
    "poisson": np.concatenate([poisson.pmf(goals, matches.FTHG.mean()),
                               poisson.pmf(goals, matches.FTAG.mean())]),
})
fig = px.bar(check, x="goals", y="observed", color="side", barmode="group",
             title="Goals per team per match: observed (bars) vs Poisson (lines)")
for side, dash in [("Home", "solid"), ("Away", "dash")]:
    s = check[check.side == side]
    fig.add_scatter(x=s.goals, y=s.poisson, mode="lines+markers", name=f"Poisson ({side})", line_dash=dash)
fig.show()

**🧠 Your turn:** In one or two sentences, how well does the Poisson curve match? Where is it off? (Hint: look at 0 goals and draws. That gap is exactly what the Dixon-Coles adjustment fixes later.)

## 3 · Fit the model
Each team gets an **attack** and a **defense** strength, plus one **home advantage** for the whole league. Recent matches count more: with the default setting, a match from one year ago counts about half as much as yesterday's.

In [ ]:
model = fit_poisson(matches)
print(f"Trained on matches up to {model.as_of:%d %b %Y}")
print(f"Home advantage: {model.home_advantage:.2f}x goals")
ratings = model.team_ratings()
ratings

In [ ]:
fig = px.scatter(ratings.reset_index(names="team"), x="attack", y="defense", text="team",
                 title="Team strength (top-right of the chart = weaker defense, so the y-axis is flipped)",
                 labels={"attack": "Attack (higher = better)", "defense": "Defense (lower = better)"})
fig.update_yaxes(autorange="reversed"); fig.update_traces(textposition="top center")
fig.show()

## 4 · Predict Big Six matches
We trained on **every** Premier League match (that's what tells us how strong the Big Six really are), and now focus the output on Big Six head-to-heads.

In [ ]:
from itertools import permutations
big_six = [t for t in BIG_SIX if t in model.teams]
fixtures = pd.DataFrame(list(permutations(big_six, 2)), columns=["HomeTeam", "AwayTeam"])
preds = model.predict_fixtures(fixtures)
preds[["home_team", "away_team", "exp_home_goals", "exp_away_goals",
       "p_home", "p_draw", "p_away", "most_likely_score"]]

In [ ]:
# Scoreline probabilities for one match. ✏️ change the teams
home, away = "Arsenal", "Man City"
mat = model.score_matrix(home, away, max_goals=5)
px.imshow(mat, text_auto=".1%", color_continuous_scale="Greens",
          labels={"x": f"{away} goals", "y": f"{home} goals", "color": "Probability"},
          title=f"{home} vs {away}: probability of each scoreline").show()
model.predict_match(home, away)

## 5 · Honest evaluation: walk-forward backtest vs the bookmakers
We replay the **last completed season** week by week. Before each week, the model is refit using only matches that had already happened, so it never sees the future.

How to read the results:
- **Log loss** and **Brier score**: lower is better. These judge the *probabilities*, which is what matters.
- **Accuracy**: how often the most likely outcome happened. It sounds intuitive but is a weak metric, because draws are rarely the top pick.
- **Pinnacle odds** are the sharpest bookmaker in the dataset. Getting close to them is a strong result; beating them consistently is very rare.

In [ ]:
last_full = season_code(current_season_start() - 1)
print("Backtesting season", last_full, "... (takes ~10-30 seconds)")
bt = backtest(matches, test_season=last_full)
results = compare(bt, train_matches=matches[matches["season"] < last_full])
results

In [ ]:
# Calibration check: when the model says "60% home win", does the home team win about 60% of the time?
bt["bucket"] = pd.cut(bt["p_home"], bins=np.arange(0, 1.01, 0.1))
calib = bt.groupby("bucket", observed=True).agg(predicted=("p_home", "mean"),
                                                 actual=("FTR", lambda s: (s == "H").mean()),
                                                 n=("FTR", "size")).reset_index(drop=True)
fig = px.scatter(calib, x="predicted", y="actual", size="n", title="Calibration: home-win probability")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line_dash="dash")
fig.show()
calib

## 6 · 🧠 Your turn (pick one before our next session)
1. **Tune the time decay.** Re-run the backtest with `xi=0.001` (slower decay) and `xi=0.004` (faster). Which gives the lowest log loss? Example: `backtest(matches, last_full, xi=0.004)`.
2. **Big Six only.** Filter `bt` to matches where both teams are in `BIG_SIX` and run `compare()` on it. Is the model better or worse on these games?
3. **Write 3 insights** in plain English (e.g. "Home advantage is worth about X% more goals"). These go straight into your README and resume bullets.

When you're done, save your work to GitHub: **File → Save a copy in GitHub** → pick your repo → path `notebooks/01_first_model.ipynb`.

**Next phase:** deploy the Streamlit app (`app/streamlit_app.py`) so you have a live link.